# 2. Panel construction

Turns the market sample into the hourly panel used in every analysis (Sections 3–6).

**Inputs**
- `finalsample.parquet`: the 50,000 markets from `sample_selection.ipynb`.
- `price_history.parquet`: hourly CLOB prices for every market
  (converted from the downloaded SQLite database on first run).

**Outputs**
- `markets.parquet`: one row per market in the final sample (used for Table 1).
- `panel.parquet`: one row per market-hour, indexed by `(market_id, t)`.

Steps:
1. Load the price history and shrink its memory footprint.
2. Clean the market sample (missing data, outcomes that did not resolve to 0 or 1).
3. Merge prices onto markets and add time-to-close variables and price bins.
4. Summary statistics (Tables 1 and 2).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("../raw")   # data/raw
DATA_DIR = Path(".")       # this notebook lives in data/pipeline, next to its data files

PRICE_DB = RAW_DIR / "price_history_backup.db"
PRICE_PATH = DATA_DIR / "price_history.parquet"
SAMPLE_PATH = DATA_DIR / "finalsample.parquet"
MARKETS_PATH = DATA_DIR / "markets.parquet"
PANEL_PATH = DATA_DIR / "panel.parquet"

N_PRICE_BINS = 20  # bins of width 0.05, used to estimate H(Y|P)

## Step 1: Price history

The CLOB price history was downloaded as a SQLite database. It is converted to Parquet once
with DuckDB, which can stream the table to disk without loading it into memory, allowing 
it to actually run on my laptop

In [2]:
def convert_price_db() -> None:
    """Copy the price_history table from the SQLite download into a Parquet file."""
    import duckdb  # only needed for this one-off conversion

    con = duckdb.connect()
    con.execute("SET memory_limit='3GB'")
    con.execute(f"PRAGMA temp_directory='{(RAW_DIR / 'duckdb_tmp').as_posix()}'")  # spill to disk, not RAM
    con.execute("INSTALL sqlite; LOAD sqlite;")
    con.execute(f"ATTACH '{PRICE_DB.as_posix()}' AS pm (TYPE sqlite)")
    con.execute(f"COPY (SELECT * FROM pm.price_history) TO '{PRICE_PATH.as_posix()}' (FORMAT parquet)")
    con.close()


def memory_mb(df: pd.DataFrame) -> pd.Series:
    """Memory used by each column, in MB."""
    return df.memory_usage(deep=True, index=False) / 1e6


if not PRICE_PATH.exists():
    convert_price_db()

prices = pd.read_parquet(PRICE_PATH)
memory_before = memory_mb(prices)

The raw table stores market and token IDs as long strings and uses 64 bit numbers throughout,
which uses way more memory than required. Since there are only ~50,000 distinct IDs, we can 
store them as a category dtype instead, cutting memory by more than 80%. Timestamps fit in 32 bits, 
and `fidelity_min` is always 60, so it is dropped.

In [3]:
def optimize_price_history(prices: pd.DataFrame) -> pd.DataFrame:
    """Shrink the price history's memory footprint and clip prices to [0, 1]."""
    assert (prices["fidelity_min"] == 60).all(), "expected hourly prices only"
    prices = prices.drop(columns="fidelity_min")

    for col in ["token_id", "market_id"]:
        prices[col] = prices[col].astype("category")

    # a handful of prices fall slightly outside [0, 1]
    out_of_range = (prices["p"] < 0) | (prices["p"] > 1)
    print(f"Prices clipped to [0, 1]: {out_of_range.sum():,}")

    prices["p"] = prices["p"].clip(0, 1).astype("float32")
    prices["t"] = prices["t"].astype("uint32")  # Unix seconds
    return prices


prices = optimize_price_history(prices)
memory_after = memory_mb(prices)

memory = pd.DataFrame({"Before (MB)": memory_before, "After (MB)": memory_after})
memory.loc["Total"] = memory.sum()
reduction = 1 - memory.loc["Total", "After (MB)"] / memory.loc["Total", "Before (MB)"]
print(f"Memory reduction: {reduction:.1%}")
memory.round(1)

Prices clipped to [0, 1]: 110
Memory reduction: 86.9%


,Before (MB),After (MB)
fidelity_min,347.2,NaN
market_id,617.6,174.3
p,347.2,173.6
t,347.2,173.6
token_id,3690.9,177.8
Total,5350.1,699.3


## Step 2: Clean the market sample

We drop markets with missing volume or close time, and markets whose final Yes price is not exactly
0 or 1. I'm assuming they did not resolve cleanly, so we won't include them in the dataset.

`negRisk` is missing for some (mostly older) markets, which predate Polymarket's negative-risk
mechanism. We treat a missing value as not negative risk.

In [4]:
MARKET_COLS = ["id", "question", "volumeNum", "createdAt", "closedTime", "endDate",
               "outcomePrices", "negRisk"]


def clean_markets(sample: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Drop markets without a clean outcome and parse types. Returns the markets and a count log."""
    markets = sample[MARKET_COLS].copy()
    log = {"Selected sample": len(markets)}

    markets = markets.dropna(subset=["volumeNum"])
    log["With volume"] = len(markets)

    markets = markets.dropna(subset=["closedTime"])
    log["With close time"] = len(markets)

    # outcomePrices is a JSON string like '["1", "0"]', ordered [Yes, No]
    yes_price = markets["outcomePrices"].map(lambda s: float(json.loads(s)[0]))
    markets = markets[yes_price.isin([0.0, 1.0])].copy()
    markets["yes_outcomes"] = yes_price[markets.index].astype(bool)
    log["Resolved cleanly to 0 or 1"] = len(markets)

    print(f"Markets with missing negRisk, treated as False: {markets['negRisk'].isna().sum():,}")
    markets["negRisk"] = markets["negRisk"].eq(True)

    for col in ["createdAt", "closedTime", "endDate"]:
        markets[col] = pd.to_datetime(markets[col], format="ISO8601", utc=True)

    markets = markets.drop(columns="outcomePrices")
    return markets, log


sample = pd.read_parquet(SAMPLE_PATH)
markets, selection_log = clean_markets(sample)

Markets with missing negRisk, treated as False: 571


## Step 3: Build the panel

Only small market-level fields are merged onto the 43-million-row price table. Text fields like
`question` would be copied onto every row, so they stay in `markets.parquet`.

The merge is an inner join, so markets with no CLOB price history drop out here.

Two time variables are added:
- `time_to_resolution`: hours until the market actually closed. Used to measure information at a
  given horizon (Section 5).
- `hours_to_end`: hours until the market's *scheduled* end date. Traders know this in advance,
  unlike the actual close time, so we can use it in the martingale regression of Section 6.

In [5]:
def build_panel(prices: pd.DataFrame, markets: pd.DataFrame) -> pd.DataFrame:
    """Merge market outcomes onto hourly prices and add time and price-bin variables."""
    market_fields = markets[["id", "yes_outcomes", "negRisk", "closedTime", "endDate"]].copy()
    # matching the price table's categorical dtype keeps market_id as a category through the
    # merge, instead of expanding it into 43 million strings
    market_fields["market_id"] = market_fields.pop("id").astype(prices["market_id"].dtype)

    panel = prices.merge(market_fields, on="market_id", how="inner")
    panel = panel.drop(columns="token_id")
    panel["market_id"] = panel["market_id"].cat.remove_unused_categories()

    panel = panel.sort_values(["market_id", "t"]).set_index(["market_id", "t"])
    assert panel.index.is_unique, "expected one price per market per hour"

    now = pd.to_datetime(panel.index.get_level_values("t"), unit="s", utc=True)

    def hours_until(when: pd.Series) -> pd.Series:
        return ((when - now).dt.total_seconds() / 3600).astype("float32")

    panel["time_to_resolution"] = hours_until(panel["closedTime"])
    panel["hours_to_end"] = hours_until(panel["endDate"])

    edges = np.linspace(0, 1, N_PRICE_BINS + 1)
    labels = [f"{lo:.2f}-{hi:.2f}" for lo, hi in zip(edges[:-1], edges[1:])]
    panel["price_bins"] = pd.cut(panel["p"], bins=edges, labels=labels, include_lowest=True)

    column_order = ["p", "price_bins", "yes_outcomes", "negRisk", "time_to_resolution", "hours_to_end"]
    return panel[column_order]


panel = build_panel(prices, markets)
del prices  # free ~700 MB

in_panel = markets["id"].isin(panel.index.get_level_values("market_id"))
markets = markets[in_panel]
selection_log["With CLOB price history"] = len(markets)

print(f"Panel: {len(panel):,} market-hours from {len(markets):,} markets")

C:\Users\daniel\AppData\Local\Temp\ipykernel_36852\616694776.py:6: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  market_fields["market_id"] = market_fields.pop("id").astype(prices["market_id"].dtype)


Panel: 43,299,348 market-hours from 49,661 markets


## Sample selection log

Counts after each cleaning step. These are the numbers reported in Section 3.1.

In [6]:
log = pd.Series(selection_log, name="Markets").to_frame()
log["Dropped"] = (-log["Markets"].diff()).fillna(0).astype(int)
log

,Markets,Dropped
Selected sample,50000,0
With volume,49986,14
With close time,49983,3
Resolved cleanly to 0 or 1,49971,12
With CLOB price history,49661,310


## Save

In [7]:
markets.to_parquet(MARKETS_PATH, index=False)
panel.to_parquet(PANEL_PATH, compression="zstd")  # zstd compresses this panel better than snappy

## Step 4: Summary statistics

Table 1 describes the final markets and Table 2 the market-hours in the panel.

In [8]:
def describe(series: pd.Series) -> dict:
    """Mean, median, SD, min and max, as reported in Tables 1 and 2."""
    return {"Mean": series.mean(), "Median": series.median(), "SD": series.std(),
            "Min": series.min(), "Max": series.max()}


duration_days = (markets["closedTime"] - markets["createdAt"]).dt.total_seconds() / 86400

table_1 = pd.DataFrame({
    "Trading volume ($ million)": describe(markets["volumeNum"] / 1e6),
    "Market duration (days)": describe(duration_days),
}).T
table_1.loc["Resolved Yes (=1)", "Mean"] = markets["yes_outcomes"].mean()
table_1.loc["Negative risk (=1)", "Mean"] = markets["negRisk"].mean()
table_1.round(3)

,Mean,Median,SD,Min,Max
Trading volume ($ million),1.131,0.161,11.018,0.052,1531.479
Market duration (days),38.856,13.378,64.476,0.088,939.485
Resolved Yes (=1),0.286,NaN,NaN,NaN,NaN
Negative risk (=1),0.624,NaN,NaN,NaN,NaN


In [9]:
table_2 = pd.DataFrame({
    "Price": describe(panel["p"]),
    "Time to resolution (days)": describe(panel["time_to_resolution"] / 24),
}).T
table_2.round(3)

,Mean,Median,SD,Min,Max
Price,0.203000,0.070000,0.270,0.0,1.000000
Time to resolution (days),70.360001,43.096001,73.388,0.0,600.578003
